# Tool-call Alignment Mismatch Result Visualization

This notebook visualizes already-computed experiment outputs. It does not retrain embedding models or machine-learning models.

Input files:

- `ratio_metrics_comparison.csv`
- `ratio_latency_comparison.csv`
- `ratio_sampling_plan.csv`
- `ratio_split_summary.csv`

The paper-facing labels use **mismatch** instead of **negative**.


## Cell 1. Install Packages


In [ ]:
%pip install pandas numpy matplotlib scikit-learn


## Cell 2. Imports and Paths


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_ROOT = ROOT / 'outputs' / 'runs' / 'paper_ratio_study_results'

METRICS_PATH = RUN_ROOT / 'ratio_metrics_comparison.csv'
LATENCY_PATH = RUN_ROOT / 'ratio_latency_comparison.csv'
SAMPLING_PATH = RUN_ROOT / 'ratio_sampling_plan.csv'
SPLIT_PATH = RUN_ROOT / 'ratio_split_summary.csv'

FIG_DIR = RUN_ROOT / 'paper_result_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RATIO_ORDER = ['1:1', '7:3', '8:2', '10:1', '100:1']
MODEL_ORDER = ['lightgbm', 'xgboost', 'random_forest', 'isolation_forest']
MODEL_LABELS = {
    'lightgbm': 'LightGBM',
    'xgboost': 'XGBoost',
    'random_forest': 'Random Forest',
    'isolation_forest': 'Isolation Forest',
}
MODEL_COLORS = {
    'lightgbm': '#1f77b4',
    'xgboost': '#ff7f0e',
    'random_forest': '#2ca02c',
    'isolation_forest': '#d62728',
}

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Run root :', RUN_ROOT)
print('Figure dir:', FIG_DIR)


## Cell 3. Load Result Tables


In [ ]:
metrics_df = pd.read_csv(METRICS_PATH)
latency_df = pd.read_csv(LATENCY_PATH)
sampling_df = pd.read_csv(SAMPLING_PATH)
split_df = pd.read_csv(SPLIT_PATH)

numeric_cols = [
    'normal_count', 'negative_count', 'dataset_positive_rate',
    'test_normal_count', 'test_negative_count', 'test_positive_rate',
    'best_validation_threshold', 'precision', 'recall', 'f1',
    'accuracy', 'auroc', 'auprc', 'tn', 'fp', 'fn', 'tp', 'fit_seconds',
]
for col in numeric_cols:
    if col in metrics_df.columns:
        metrics_df[col] = pd.to_numeric(metrics_df[col], errors='coerce')

for col in latency_df.columns:
    if col not in ['ratio_name', 'ratio_label', 'model_name']:
        latency_df[col] = pd.to_numeric(latency_df[col], errors='coerce')

for col in sampling_df.columns:
    if col not in ['ratio_name', 'ratio_label', 'sample_path']:
        sampling_df[col] = pd.to_numeric(sampling_df[col], errors='coerce')

for col in split_df.columns:
    if col not in ['ratio_name', 'ratio_label', 'split']:
        split_df[col] = pd.to_numeric(split_df[col], errors='coerce')

metrics_df['ratio_label'] = pd.Categorical(metrics_df['ratio_label'], categories=RATIO_ORDER, ordered=True)
latency_df['ratio_label'] = pd.Categorical(latency_df['ratio_label'], categories=RATIO_ORDER, ordered=True)
sampling_df['ratio_label'] = pd.Categorical(sampling_df['ratio_label'], categories=RATIO_ORDER, ordered=True)
split_df['ratio_label'] = pd.Categorical(split_df['ratio_label'], categories=RATIO_ORDER, ordered=True)

metrics_df['model_label'] = metrics_df['model_name'].map(MODEL_LABELS)
latency_df['model_label'] = latency_df['model_name'].map(MODEL_LABELS)

metrics_df = metrics_df.sort_values(['ratio_label', 'model_name']).reset_index(drop=True)
latency_df = latency_df.sort_values(['ratio_label', 'model_name']).reset_index(drop=True)
sampling_df = sampling_df.sort_values('ratio_label').reset_index(drop=True)
split_df = split_df.sort_values(['ratio_label', 'split']).reset_index(drop=True)

print('Metrics shape :', metrics_df.shape)
print('Latency shape :', latency_df.shape)
print('Sampling shape:', sampling_df.shape)
print('Split shape   :', split_df.shape)
display(metrics_df.head())


## Cell 4. Paper Summary Tables


In [ ]:
paper_metric_cols = [
    'ratio_label', 'model_label', 'precision', 'recall', 'f1',
    'accuracy', 'auroc', 'auprc', 'tn', 'fp', 'fn', 'tp',
]
paper_metrics_df = metrics_df[paper_metric_cols].copy()
paper_metrics_df = paper_metrics_df.rename(columns={
    'ratio_label': 'Normal:Mismatch Ratio',
    'model_label': 'Model',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1-score',
    'accuracy': 'Accuracy',
    'auroc': 'AUROC',
    'auprc': 'AUPRC',
    'tn': 'TN',
    'fp': 'FP',
    'fn': 'FN',
    'tp': 'TP',
})

best_by_ratio_df = (
    metrics_df.sort_values(['ratio_label', 'f1', 'auprc'], ascending=[True, False, False])
    .groupby('ratio_label', observed=True)
    .head(1)
    .reset_index(drop=True)
)
paper_best_df = best_by_ratio_df[[
    'ratio_label', 'model_label', 'precision', 'recall', 'f1',
    'auroc', 'auprc', 'test_positive_rate',
]].copy()
paper_best_df = paper_best_df.rename(columns={
    'ratio_label': 'Normal:Mismatch Ratio',
    'model_label': 'Best Model',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1-score',
    'auroc': 'AUROC',
    'auprc': 'AUPRC',
    'test_positive_rate': 'Test Mismatch Rate',
})

paper_metrics_path = FIG_DIR / 'paper_metrics_table.csv'
paper_best_path = FIG_DIR / 'paper_best_by_ratio_table.csv'
paper_metrics_df.to_csv(paper_metrics_path, index=False, encoding='utf-8-sig')
paper_best_df.to_csv(paper_best_path, index=False, encoding='utf-8-sig')

print('Paper metrics table:', paper_metrics_path)
print('Best-by-ratio table:', paper_best_path)
display(paper_best_df.round(4))
display(paper_metrics_df.round(4))


## Cell 5. Main Ratio Sensitivity Curves


In [ ]:
def get_metric_series(model_name, metric_name):
    return (
        metrics_df[metrics_df['model_name'] == model_name]
        .set_index('ratio_label')
        .reindex(RATIO_ORDER)[metric_name]
        .to_numpy(dtype=float)
    )


ratio_x = np.arange(len(RATIO_ORDER))
metric_specs = [
    ('auroc', 'AUROC'),
    ('auprc', 'AUPRC'),
    ('f1', 'F1-score'),
    ('recall', 'Recall'),
]

fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.6), sharex=True)
axes = axes.ravel()

for ax, (metric_name, metric_label) in zip(axes, metric_specs):
    for model_name in MODEL_ORDER:
        ax.plot(
            ratio_x,
            get_metric_series(model_name, metric_name),
            marker='o',
            linewidth=2,
            markersize=5,
            color=MODEL_COLORS[model_name],
            label=MODEL_LABELS[model_name],
        )
    ax.set_title(metric_label)
    ax.set_xticks(ratio_x)
    ax.set_xticklabels(RATIO_ORDER)
    ax.set_xlabel('Normal:Mismatch Ratio')
    ax.set_ylabel(metric_label)
    ax.set_ylim(0.0, 1.03)
    ax.legend(loc='lower left')

fig.suptitle('Class-Ratio Sensitivity of Tool-call Mismatch Detection', y=1.02, fontsize=15, fontweight='bold')
fig.tight_layout()
ratio_curve_path = FIG_DIR / 'figure_ratio_sensitivity_metrics.png'
ratio_curve_svg_path = FIG_DIR / 'figure_ratio_sensitivity_metrics.svg'
fig.savefig(ratio_curve_path, bbox_inches='tight')
fig.savefig(ratio_curve_svg_path, bbox_inches='tight')
plt.show()

print('Saved:', ratio_curve_path)
print('Saved:', ratio_curve_svg_path)


## Cell 6. AUPRC with Mismatch-Rate Baseline


In [ ]:
baseline_df = (
    metrics_df[['ratio_label', 'test_positive_rate']]
    .drop_duplicates()
    .set_index('ratio_label')
    .reindex(RATIO_ORDER)
)

fig, ax = plt.subplots(figsize=(9.5, 5.4))
for model_name in MODEL_ORDER:
    ax.plot(
        ratio_x,
        get_metric_series(model_name, 'auprc'),
        marker='o',
        linewidth=2,
        color=MODEL_COLORS[model_name],
        label=MODEL_LABELS[model_name],
    )

ax.plot(
    ratio_x,
    baseline_df['test_positive_rate'].to_numpy(dtype=float),
    marker='s',
    linestyle='--',
    linewidth=2,
    color='black',
    label='Mismatch-rate baseline',
)
ax.set_xticks(ratio_x)
ax.set_xticklabels(RATIO_ORDER)
ax.set_xlabel('Normal:Mismatch Ratio')
ax.set_ylabel('AUPRC')
ax.set_ylim(0.0, 1.03)
ax.set_title('AUPRC under Class Imbalance')
ax.legend(loc='upper right')
fig.tight_layout()

auprc_path = FIG_DIR / 'figure_auprc_with_mismatch_rate_baseline.png'
auprc_svg_path = FIG_DIR / 'figure_auprc_with_mismatch_rate_baseline.svg'
fig.savefig(auprc_path, bbox_inches='tight')
fig.savefig(auprc_svg_path, bbox_inches='tight')
plt.show()

print('Saved:', auprc_path)
print('Saved:', auprc_svg_path)


## Cell 7. Balanced 1:1 Performance Bar Chart


In [ ]:
balanced_df = metrics_df[metrics_df['ratio_label'] == '1:1'].copy()
balanced_df['model_name'] = pd.Categorical(balanced_df['model_name'], categories=MODEL_ORDER, ordered=True)
balanced_df = balanced_df.sort_values('model_name')

bar_metrics = ['precision', 'recall', 'f1', 'auroc', 'auprc']
bar_labels = ['Precision', 'Recall', 'F1-score', 'AUROC', 'AUPRC']
x = np.arange(len(bar_metrics))
width = 0.18

fig, ax = plt.subplots(figsize=(11, 5.8))
for idx, model_name in enumerate(MODEL_ORDER):
    row = balanced_df[balanced_df['model_name'] == model_name].iloc[0]
    values = [row[m] for m in bar_metrics]
    offset = (idx - 1.5) * width
    ax.bar(
        x + offset,
        values,
        width=width,
        color=MODEL_COLORS[model_name],
        label=MODEL_LABELS[model_name],
    )

ax.set_xticks(x)
ax.set_xticklabels(bar_labels)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Score')
ax.set_title('Model Performance on the 1:1 Balanced Dataset')
ax.legend(ncol=2, loc='lower right')
fig.tight_layout()

balanced_bar_path = FIG_DIR / 'figure_balanced_1_1_model_performance.png'
balanced_bar_svg_path = FIG_DIR / 'figure_balanced_1_1_model_performance.svg'
fig.savefig(balanced_bar_path, bbox_inches='tight')
fig.savefig(balanced_bar_svg_path, bbox_inches='tight')
plt.show()

print('Saved:', balanced_bar_path)
print('Saved:', balanced_bar_svg_path)


## Cell 8. Latency Comparison


In [ ]:
latency_summary_df = (
    latency_df.groupby(['model_name', 'model_label'], as_index=False)
    .agg(
        model_inference_ms_mean=('model_inference_ms_mean', 'mean'),
        model_inference_ms_p95=('model_inference_ms_p95', 'mean'),
        end_to_end_ms_mean=('end_to_end_ms_mean', 'mean'),
        end_to_end_ms_p95=('end_to_end_ms_p95', 'mean'),
    )
)
latency_summary_df['model_name'] = pd.Categorical(latency_summary_df['model_name'], categories=MODEL_ORDER, ordered=True)
latency_summary_df = latency_summary_df.sort_values('model_name')

fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.2))

axes[0].bar(
    latency_summary_df['model_label'],
    latency_summary_df['model_inference_ms_mean'],
    color=[MODEL_COLORS[m] for m in latency_summary_df['model_name']],
)
axes[0].set_title('Model Inference Latency')
axes[0].set_ylabel('Mean latency (ms)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(
    latency_summary_df['model_label'],
    latency_summary_df['end_to_end_ms_mean'],
    color=[MODEL_COLORS[m] for m in latency_summary_df['model_name']],
)
axes[1].set_title('End-to-End Latency')
axes[1].set_ylabel('Mean latency (ms)')
axes[1].tick_params(axis='x', rotation=20)

fig.suptitle('Average Latency Across Ratio Settings', y=1.03, fontsize=15, fontweight='bold')
fig.tight_layout()

latency_path = FIG_DIR / 'figure_latency_comparison.png'
latency_svg_path = FIG_DIR / 'figure_latency_comparison.svg'
fig.savefig(latency_path, bbox_inches='tight')
fig.savefig(latency_svg_path, bbox_inches='tight')
plt.show()

latency_table_path = FIG_DIR / 'paper_latency_summary_table.csv'
latency_summary_df.to_csv(latency_table_path, index=False, encoding='utf-8-sig')

print('Saved:', latency_path)
print('Saved:', latency_svg_path)
print('Saved:', latency_table_path)
display(latency_summary_df.round(4))


## Cell 9. Confusion Matrix for Best Model by Ratio


In [ ]:
best_cm_df = best_by_ratio_df[['ratio_label', 'model_label', 'tn', 'fp', 'fn', 'tp']].copy()
best_cm_df = best_cm_df.rename(columns={
    'ratio_label': 'Normal:Mismatch Ratio',
    'model_label': 'Best Model',
    'tn': 'TN',
    'fp': 'FP',
    'fn': 'FN',
    'tp': 'TP',
})

best_cm_path = FIG_DIR / 'paper_best_model_confusion_matrix_table.csv'
best_cm_df.to_csv(best_cm_path, index=False, encoding='utf-8-sig')
display(best_cm_df)

fig, axes = plt.subplots(1, len(best_by_ratio_df), figsize=(15.5, 3.3))
if len(best_by_ratio_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, best_by_ratio_df.iterrows()):
    cm = np.array([[row['tn'], row['fp']], [row['fn'], row['tp']]], dtype=float)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f"{row['ratio_label']}/n{row['model_label']}")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['True 0', 'True 1'])
    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{int(cm[i, j]):,}/n({cm_norm[i, j]:.2f})",
                ha='center',
                va='center',
                color='black' if cm_norm[i, j] < 0.65 else 'white',
                fontsize=9,
            )

fig.suptitle('Confusion Matrix of the Best-F1 Model by Ratio', y=1.05, fontsize=14, fontweight='bold')
fig.tight_layout()
cm_path = FIG_DIR / 'figure_best_model_confusion_matrices.png'
cm_svg_path = FIG_DIR / 'figure_best_model_confusion_matrices.svg'
fig.savefig(cm_path, bbox_inches='tight')
fig.savefig(cm_svg_path, bbox_inches='tight')
plt.show()

print('Saved:', best_cm_path)
print('Saved:', cm_path)
print('Saved:', cm_svg_path)


## Cell 10. Optional ROC/PR Curves for a Selected Ratio


In [ ]:
SELECTED_RATIO_NAME = '1_1'  # one of: 1_1, 7_3, 8_2, 10_1, 100_1
SELECTED_RATIO_LABEL = SELECTED_RATIO_NAME.replace('_', ':', 1)

try:
    from sklearn.metrics import precision_recall_curve, roc_curve, average_precision_score, roc_auc_score
except Exception as exc:
    raise RuntimeError('Install scikit-learn to draw ROC/PR curves from test_predictions.csv') from exc

prediction_frames = {}
for model_name in MODEL_ORDER:
    pred_path = RUN_ROOT / f'ratio_{SELECTED_RATIO_NAME}' / model_name / 'test_predictions.csv'
    if pred_path.exists():
        prediction_frames[model_name] = pd.read_csv(pred_path)
    else:
        print('Missing:', pred_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))

for model_name, pred_df in prediction_frames.items():
    y_true = pred_df['label'].to_numpy(dtype=int)
    y_score = pred_df['score'].to_numpy(dtype=float)

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auroc = roc_auc_score(y_true, y_score)
    axes[0].plot(
        fpr,
        tpr,
        linewidth=2,
        color=MODEL_COLORS[model_name],
        label=f'{MODEL_LABELS[model_name]} ({auroc:.4f})',
    )

    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    axes[1].plot(
        recall_curve,
        precision_curve,
        linewidth=2,
        color=MODEL_COLORS[model_name],
        label=f'{MODEL_LABELS[model_name]} ({auprc:.4f})',
    )

axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
axes[0].set_title(f'ROC Curve ({SELECTED_RATIO_LABEL})')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

selected_positive_rate = prediction_frames[next(iter(prediction_frames))]['label'].mean()
axes[1].axhline(selected_positive_rate, linestyle='--', color='gray', linewidth=1, label='Mismatch-rate baseline')
axes[1].set_title(f'Precision-Recall Curve ({SELECTED_RATIO_LABEL})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')

fig.tight_layout()
curve_path = FIG_DIR / f'figure_roc_pr_curves_ratio_{SELECTED_RATIO_NAME}.png'
curve_svg_path = FIG_DIR / f'figure_roc_pr_curves_ratio_{SELECTED_RATIO_NAME}.svg'
fig.savefig(curve_path, bbox_inches='tight')
fig.savefig(curve_svg_path, bbox_inches='tight')
plt.show()

print('Saved:', curve_path)
print('Saved:', curve_svg_path)


## Cell 11. Output Files


In [ ]:
print('Figure directory:', FIG_DIR)
for path in sorted(FIG_DIR.glob('*')):
    print(path.name)
